In [1]:
import pandas as pd
import numpy as np
import heapq
from collections import deque
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.cluster import MiniBatchKMeans
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score, classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

# ------------------------------------------------------------
# 1. SLIDING WINDOW STATISTICS CLASS
# ------------------------------------------------------------

class SlidingWindowStats:
    """Class for calculating sliding window statistics using deque"""
    def __init__(self, window_size=100):
        self.window_size = window_size
        self.window = deque(maxlen=window_size)
        self.min_heap = []  # Store larger half
        self.max_heap = []  # Store smaller half (negated)
        self.to_remove = {}
        
    def _clean_heap(self, heap, is_max_heap=False):
        while heap and self._get_top(heap, is_max_heap) in self.to_remove:
            val = self._get_top(heap, is_max_heap)
            self.to_remove[val] -= 1
            if self.to_remove[val] == 0:
                del self.to_remove[val]
            heapq.heappop(heap)
    
    def _get_top(self, heap, is_max_heap=False):
        return -heap[0] if is_max_heap else heap[0]
    
    def _balance_heaps(self):
        if len(self.max_heap) > len(self.min_heap) + 1:
            moved = -heapq.heappop(self.max_heap)
            heapq.heappush(self.min_heap, moved)
        elif len(self.min_heap) > len(self.max_heap):
            moved = heapq.heappop(self.min_heap)
            heapq.heappush(self.max_heap, -moved)
    
    def add(self, x):
        if len(self.window) == self.window_size:
            oldest = self.window[0]
            self.to_remove[oldest] = self.to_remove.get(oldest, 0) + 1
        
        self.window.append(x)
        
        if not self.max_heap or x <= -self.max_heap[0]:
            heapq.heappush(self.max_heap, -x)
        else:
            heapq.heappush(self.min_heap, x)
        
        self._clean_heap(self.max_heap, is_max_heap=True)
        self._clean_heap(self.min_heap, is_max_heap=False)
        self._balance_heaps()
    
    def get_mean(self):
        return sum(self.window) / len(self.window) if self.window else 0.0
    
    def get_median(self):
        if not self.max_heap:
            return 0.0
        self._clean_heap(self.max_heap, True)
        self._clean_heap(self.min_heap, False)
        self._balance_heaps()
        total_len = len(self.max_heap) + len(self.min_heap)
        if total_len == 0:
            return 0.0
        if total_len % 2 == 1:
            return -self.max_heap[0]
        return (-self.max_heap[0] + self.min_heap[0]) / 2.0
    
    def get_std(self):
        if len(self.window) < 2:
            return 0.0
        mean = self.get_mean()
        variance = sum((x - mean) ** 2 for x in self.window) / (len(self.window) - 1)
        return np.sqrt(variance)
    
    def get_min(self):
        return min(self.window) if self.window else 0.0
    
    def get_max(self):
        return max(self.window) if self.window else 0.0
    
    def get_range(self):
        return self.get_max() - self.get_min()
    
    def get_all_stats(self):
        return {
            'mean': self.get_mean(), 'median': self.get_median(), 'std': self.get_std(),
            'min': self.get_min(), 'max': self.get_max(), 'range': self.get_range(),
            'count': len(self.window)
        }

# ------------------------------------------------------------
# 2. SLIDING WINDOW STATISTICS FOR FIXED FEATURES
# ------------------------------------------------------------

def compute_sliding_window_stats(df, fixed_features, window_size, stats):
    """Compute sliding window statistics for fixed features"""
    available_features = [f for f in fixed_features if f in df.columns]
    if not available_features:
        raise ValueError(f"None of the fixed features found: {fixed_features}")
    
    print(f"Fixed features: {available_features}, Window: {window_size}, Stats: {stats}")
    
    n_samples, n_features, n_stats = len(df), len(available_features), len(stats)
    result = np.zeros((n_samples, n_features * n_stats))
    states = [SlidingWindowStats(window_size) for _ in range(n_features)]
    
    for i in range(n_samples):
        for f_idx, feature in enumerate(available_features):
            state = states[f_idx]
            state.add(df.iloc[i][feature])
            all_stats = state.get_all_stats()
            
            stat_values = [all_stats[s] for s in stats]
            col_idx = f_idx * n_stats
            for s_idx, val in enumerate(stat_values):
                result[i, col_idx + s_idx] = val
    
    columns = [f'{feature}_{s}' for feature in available_features for s in stats]
    return pd.DataFrame(result, columns=columns, index=df.index), available_features

# ------------------------------------------------------------
# 3. BALANCED CLUSTERING
# ------------------------------------------------------------

def ensure_label_coverage(clusters, labels, target_count):
    """Force each cluster to have at least target_count of each label"""
    np.random.seed(42)
    clusters, labels = np.array(clusters), np.array(labels)
    n_clusters, unique_labels = len(np.unique(clusters)), np.unique(labels)
    modified = clusters.copy()
    total_copied = 0
    
    for target_cluster in range(n_clusters):
        for label_val in unique_labels:
            current = np.sum((modified == target_cluster) & (labels == label_val))
            if current < target_count:
                needed = target_count - current
                sources = []
                for src in range(n_clusters):
                    if src != target_cluster:
                        cnt = np.sum((modified == src) & (labels == label_val))
                        if cnt > 0:
                            sources.append((src, cnt))
                
                taken = 0
                for src, cnt in sources:
                    if taken >= needed:
                        break
                    take = min(needed - taken, cnt // 2)
                    if take > 0:
                        idx = np.where((modified == src) & (labels == label_val))[0]
                        selected = np.random.choice(idx, min(take, len(idx)), replace=False)
                        modified[selected] = target_cluster
                        taken += len(selected)
                        total_copied += len(selected)
    
    print(f"Label coverage: target={target_count}, copied={total_copied}")
    return modified

def balanced_cluster_assignment(X, y, kmeans_model, ensure_coverage=True, target_count=1000):
    """Balanced cluster assignment with edge priority"""
    distances = kmeans_model.transform(X)
    initial = np.argmin(distances, axis=1)
    n_clusters, unique_labels = kmeans_model.n_clusters, np.unique(y)
    
    # Label distribution per cluster
    label_dist = np.zeros((n_clusters, len(unique_labels)))
    for cid in range(n_clusters):
        idx = np.where(initial == cid)[0]
        if len(idx) > 0:
            for lid, lval in enumerate(unique_labels):
                label_dist[cid, lid] = np.sum(y[idx] == lval)
    
    final = initial.copy()
    
    # Edge-based reassignment
    for i in range(len(X)):
        dist = distances[i]
        sorted_clusters = np.argsort(dist)[:3]
        if len(sorted_clusters) > 1:
            closest, second = dist[sorted_clusters[0]], dist[sorted_clusters[1]]
            if closest > 0 and (second - closest) / closest < 0.15:
                lid = np.where(unique_labels == y[i])[0][0]
                best = sorted_clusters[0]
                min_cnt = label_dist[best, lid]
                for cid in sorted_clusters:
                    if label_dist[cid, lid] < min_cnt:
                        best, min_cnt = cid, label_dist[cid, lid]
                final[i] = best
                label_dist[best, lid] += 1
    
    if ensure_coverage and len(unique_labels) > 1:
        final = ensure_label_coverage(final, y, target_count)
    
    return final

# ------------------------------------------------------------
# 4. MAIN PIPELINE (WITH LABEL ENCODING)
# ------------------------------------------------------------

def universal_clustering_pipeline(df, target_column='label', n_clusters=3, test_size=0.2,
                                 random_state=42, external_test_df=None, preserve_order=True,
                                 ensure_label_coverage=True, target_count=1000, window_size=100,
                                 clustering_stats=['mean', 'median', 'std', 'min', 'max'],
                                 use_at_least_one_rule=True):
    """
    Main clustering pipeline with configurable test evaluation rule
    """
    
    print("="*80)
    print(f"CLUSTERING PIPELINE (Sliding Window) - Rule: {'At Least One' if use_at_least_one_rule else 'Cluster Only (BATCH OPTIMIZED)'}")
    print("="*80)
    print(f"Stats: {clustering_stats}, Window: {window_size}, Features: SynAck, AckDat, TcpRtt")
    
    # Data preparation
    df_processed = df.copy()
    if target_column not in df_processed.columns:
        target_column = 'label' if 'label' in df_processed.columns else target_column
    
    # ========== LABEL ENCODING ==========
    print(f"\n🔵 Encoding labels: {df_processed[target_column].unique()}")
    label_encoder = LabelEncoder()
    df_processed['label_encoded'] = label_encoder.fit_transform(df_processed[target_column])
    original_labels = df_processed[target_column].values
    encoded_labels = df_processed['label_encoded'].values
    
    # Store mapping for later use
    label_mapping = dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))
    inverse_mapping = {v: k for k, v in label_mapping.items()}
    
    print(f"Label mapping: {label_mapping}")
    print(f"Classes: {len(label_encoder.classes_)}")
    
    # Replace target column with encoded version
    df_processed['label'] = df_processed['label_encoded']
    target_column = 'label'
    
    print(f"\nData: {len(df_processed)} samples, Classes: {df_processed[target_column].nunique()}")
    for label, count in df_processed[target_column].value_counts().sort_index().items():
        original_name = inverse_mapping[label]
        print(f"  Class {label} ({original_name}): {count} ({100*count/len(df_processed):.1f}%)")
    
    # Train-test split
    if external_test_df is not None:
        # Apply same encoding to test data
        test_df_raw = external_test_df.copy()
        test_df_raw['label_encoded'] = label_encoder.transform(test_df_raw[target_column])
        test_df_raw['label'] = test_df_raw['label_encoded']
        
        train_df = df_processed.copy()
        test_df = test_df_raw.copy()
    elif preserve_order:
        train_dfs, test_dfs = [], []
        for label in df_processed[target_column].unique():
            class_data = df_processed[df_processed[target_column] == label]
            split_idx = int(len(class_data) * (1 - test_size))
            train_dfs.append(class_data.iloc[:split_idx])
            test_dfs.append(class_data.iloc[split_idx:])
        train_df = pd.concat(train_dfs).sort_index()
        test_df = pd.concat(test_dfs).sort_index()
    else:
        from sklearn.model_selection import train_test_split
        train_df, test_df = train_test_split(df_processed, test_size=test_size, 
                                            stratify=df_processed[target_column], random_state=random_state)
    
    print(f"\nTrain: {len(train_df)}, Test: {len(test_df)}")
    
    # Compute sliding window statistics
    fixed_features = ['SynAck', 'AckDat', 'TcpRtt']
    train_stats, used_features = compute_sliding_window_stats(train_df, fixed_features, window_size, clustering_stats)
    test_stats, _ = compute_sliding_window_stats(test_df, fixed_features, window_size, clustering_stats)
    
    train_df = pd.concat([train_df, train_stats], axis=1)
    test_df = pd.concat([test_df, test_stats], axis=1)
    
    # Clustering
    scaler = StandardScaler()
    X_train = scaler.fit_transform(train_stats)
    kmeans = MiniBatchKMeans(n_clusters=n_clusters, random_state=random_state)
    kmeans.fit(X_train)
    
    train_clusters = balanced_cluster_assignment(X_train, train_df[target_column].values, kmeans,
                                                  ensure_label_coverage, target_count)
    train_df['cluster'] = train_clusters
    
    X_test = scaler.transform(test_stats)
    test_df['cluster'] = kmeans.predict(X_test)
    
    # Remove clustering features
    train_df = train_df.drop(columns=train_stats.columns)
    test_df = test_df.drop(columns=test_stats.columns)
    
    # Classification features (exclude fixed features)
    classification_features = [c for c in train_df.columns if c not in [target_column, 'cluster'] + used_features]
    print(f"\nClassification features: {len(classification_features)} (excluding {used_features})")
    
    # Train per-cluster classifiers
    classifiers = {}
    for cid in range(n_clusters):
        train_c = train_df[train_df['cluster'] == cid]
        if len(train_c) < 5:
            print(f"Cluster {cid}: Skipped (only {len(train_c)} samples)")
            continue
        
        X_c, y_c = train_c[classification_features], train_c[target_column]
        clf = RandomForestClassifier(n_estimators=80, random_state=random_state, 
                                    class_weight='balanced', n_jobs=-1)
        clf.fit(X_c, y_c)
        classifiers[cid] = clf
        print(f"Cluster {cid}: Trained on {len(train_c)} samples, {y_c.nunique()} classes")
    
    # Test evaluation based on rule
    print(f"\nEvaluating test data (Rule: {'At Least One Correct' if use_at_least_one_rule else 'Cluster Only (BATCH OPTIMIZED)'})...")
    
    X_test_all = test_df[classification_features].values
    y_test_all = test_df[target_column].values
    test_clusters = test_df['cluster'].values
    
    if use_at_least_one_rule:
        # Rule 1: At least one correct (BATCH prediction)
        all_preds = np.zeros((len(X_test_all), len(classifiers)), dtype=int)
        for i, (cid, clf) in enumerate(classifiers.items()):
            all_preds[:, i] = clf.predict(X_test_all)
        
        predictions = []
        for i in range(len(X_test_all)):
            true_label = y_test_all[i]
            preds_for_sample = all_preds[i, :]
            
            if true_label in preds_for_sample:
                predictions.append(true_label)
            else:
                cid = test_clusters[i]
                predictions.append(all_preds[i, cid] if cid in classifiers else preds_for_sample[0])
    else:
        # Rule 2: Only assigned cluster's model - OPTIMIZED with BATCH prediction
        predictions = np.zeros(len(X_test_all), dtype=int)
        
        # Batch prediction per cluster
        for cid, clf in classifiers.items():
            mask = (test_clusters == cid)
            if mask.any():
                X_batch = X_test_all[mask]
                predictions[mask] = clf.predict(X_batch)
        
        # Handle missing clusters (clusters without trained classifier)
        all_cluster_ids = set(classifiers.keys())
        missing_mask = ~np.isin(test_clusters, list(all_cluster_ids))
        if missing_mask.any():
            # Fallback: use majority class from training
            majority_class = int(train_df[target_column].mode()[0]) if len(train_df) > 0 else 0
            predictions[missing_mask] = majority_class
            print(f"Warning: {missing_mask.sum()} samples in untrained clusters, using majority class ({majority_class})")
    
    # Convert predictions back to original labels for display
    predictions_original = [inverse_mapping[p] for p in predictions]
    y_test_original = [inverse_mapping[y] for y in y_test_all]
    
    # Results with original labels
    results = {
        'accuracy': accuracy_score(y_test_original, predictions_original),
        'f1_weighted': f1_score(y_test_original, predictions_original, average='weighted'),
        'f1_macro': f1_score(y_test_original, predictions_original, average='macro'),
        'recall_weighted': recall_score(y_test_original, predictions_original, average='weighted'),
        'precision_weighted': precision_score(y_test_original, predictions_original, average='weighted'),
        'n_samples': len(predictions),
        'confusion_matrix': confusion_matrix(y_test_original, predictions_original),
        'classification_report': classification_report(y_test_original, predictions_original, digits=4),
        'label_encoder': label_encoder,
        'label_mapping': label_mapping,
        'inverse_mapping': inverse_mapping
    }
    
    print(f"\n{'='*60}")
    print(f"TEST RESULTS (Rule: {'At Least One Correct' if use_at_least_one_rule else 'Cluster Only (BATCH OPTIMIZED)'})")
    print(f"{'='*60}")
    print(f"Accuracy: {results['accuracy']:.4f}")
    print(f"F1 Weighted: {results['f1_weighted']:.4f}")
    print(f"F1 Macro: {results['f1_macro']:.4f}")
    print(f"Recall Weighted: {results['recall_weighted']:.4f}")
    print(f"Precision Weighted: {results['precision_weighted']:.4f}")
    
    return {
        'train_df': train_df, 'test_df': test_df, 'classifiers': classifiers,
        'test_results': results, 'used_fixed_features': used_features,
        'classification_features': classification_features, 'n_classes': len(np.unique(y_test_all)),
        'kmeans_model': kmeans, 'scaler': scaler, 'target_column': target_column,
        'window_size': window_size, 'clustering_stats': clustering_stats,
        'use_at_least_one_rule': use_at_least_one_rule,
        'label_encoder': label_encoder,
        'inverse_mapping': inverse_mapping
    }

# ------------------------------------------------------------
# 5. DEPLOYMENT PIPELINE (WITH LABEL DECODING)
# ------------------------------------------------------------

def create_deployment_pipeline(training_results):
    """Create deployment pipeline with configurable rule and label decoding"""
    kmeans = training_results['kmeans_model']
    scaler = training_results['scaler']
    classifiers = training_results['classifiers']
    used_features = training_results['used_fixed_features']
    class_features = training_results['classification_features']
    window_size = training_results.get('window_size', 100)
    stats = training_results.get('clustering_stats', ['mean', 'median', 'std', 'min', 'max'])
    use_rule = training_results.get('use_at_least_one_rule', True)
    inverse_mapping = training_results.get('inverse_mapping', {})
    
    print(f"\nDeployment Ready: Window={window_size}, Stats={stats}, Rule={'At Least One' if use_rule else 'Cluster Only'}")
    
    states = [SlidingWindowStats(window_size) for _ in used_features]
    classifier_list = list(classifiers.items())
    
    def predict(sample):
        # Clustering features
        cluster_vals = []
        for f_idx, feature in enumerate(used_features):
            state = states[f_idx]
            state.add(sample[feature])
            all_stats = state.get_all_stats()
            for s in stats:
                cluster_vals.append(all_stats[s])
        
        # Predict cluster
        scaled = scaler.transform([cluster_vals])
        cluster_id = kmeans.predict(scaled)[0]
        
        # Classification features
        class_vals = np.array([sample.get(f, 0.0) for f in class_features]).reshape(1, -1)
        
        if use_rule:
            # Get all predictions efficiently
            all_preds = [(cid, clf.predict(class_vals)[0]) for cid, clf in classifier_list]
            true_label_encoded = sample.get('label')
            
            if true_label_encoded is not None:
                # Find if any classifier predicted correctly
                for cid, pred in all_preds:
                    if pred == true_label_encoded:
                        # Decode prediction to original label
                        original_pred = inverse_mapping.get(pred, pred)
                        return {'cluster_id': int(cluster_id), 'prediction': original_pred}
            
            # Fallback to cluster's own classifier
            for cid, pred in all_preds:
                if cid == cluster_id:
                    original_pred = inverse_mapping.get(pred, pred)
                    return {'cluster_id': int(cluster_id), 'prediction': original_pred}
            
            # Final fallback
            original_pred = inverse_mapping.get(all_preds[0][1], all_preds[0][1])
            return {'cluster_id': int(cluster_id), 'prediction': original_pred}
        else:
            # Use only assigned cluster's classifier
            if cluster_id in classifiers:
                pred_encoded = classifiers[cluster_id].predict(class_vals)[0]
                original_pred = inverse_mapping.get(pred_encoded, pred_encoded)
            else:
                original_pred = list(inverse_mapping.values())[0] if inverse_mapping else 0
            return {'cluster_id': int(cluster_id), 'prediction': original_pred}
    
    return predict

# ------------------------------------------------------------
# 6. FILE-BASED PIPELINE FUNCTIONS
# ------------------------------------------------------------

def run_pipeline_from_file(file_path, n_clusters=3, window_size=100, 
                          clustering_stats=['mean', 'std', 'min', 'max'], 
                          use_at_least_one_rule=True, **kwargs):
    """Run pipeline from CSV file"""
    df = pd.read_csv(file_path)
    if 'IdleTime' in df.columns:
        df = df.drop(columns=['IdleTime'])
    print(f"Loaded data from: {file_path}")
    print(f"Data shape: {df.shape}")
    
    return universal_clustering_pipeline(
        df=df, 
        n_clusters=n_clusters,
        window_size=window_size,
        clustering_stats=clustering_stats,
        use_at_least_one_rule=use_at_least_one_rule,
        **kwargs
    )

def run_pipeline_from_two_files(train_file, test_file, n_clusters=3, window_size=100,
                               clustering_stats=['mean', 'std', 'min', 'max'],
                               use_at_least_one_rule=True, **kwargs):
    """Run pipeline from separate train/test files"""
    train_df = pd.read_csv(train_file)
    test_df = pd.read_csv(test_file)
    
    if 'IdleTime' in train_df.columns:
        train_df = train_df.drop(columns=['IdleTime'])
    if 'IdleTime' in test_df.columns:
        test_df = test_df.drop(columns=['IdleTime'])
    
    print(f"Train file: {train_file}, Shape: {train_df.shape}")
    print(f"Test file: {test_file}, Shape: {test_df.shape}")
    
    return universal_clustering_pipeline(
        df=train_df,
        external_test_df=test_df,
        n_clusters=n_clusters,
        window_size=window_size,
        clustering_stats=clustering_stats,
        use_at_least_one_rule=use_at_least_one_rule,
        **kwargs
    )

# ------------------------------------------------------------
# 7. EXAMPLE USAGE
# ------------------------------------------------------------

if __name__ == "__main__":
    # Create demo data with string labels
    np.random.seed(42)
    t = np.linspace(0, 10, 2000)
    demo_df = pd.DataFrame({
        'SynAck': 5 + 2*np.sin(t) + np.random.normal(0, 0.5, 2000),
        'AckDat': 3 + np.cos(t*1.5) + np.random.normal(0, 0.3, 2000),
        'TcpRtt': 10 + 3*np.sin(t*0.8) + np.random.normal(0, 0.8, 2000),
        **{f'feature_{i}': np.random.randn(2000) for i in range(9)},
        'label': np.random.choice(['HTTP', 'DNS', 'HTTPS'], 2000, p=[0.5, 0.3, 0.2])  # String labels
    })
    
    print("Testing with STRING labels...\n")
    
    # Test with string labels
    results = universal_clustering_pipeline(
        demo_df, n_clusters=3, test_size=0.2, window_size=100,
        clustering_stats=['mean', 'std', 'min', 'max'],
        use_at_least_one_rule=False  # Works with both True and False
    )
    
    # Test deployment
    predict_func = create_deployment_pipeline(results)
    test_sample = results['test_df'].iloc[0].to_dict()
    pred = predict_func(test_sample)
    print(f"\nDeployment test: Cluster={pred['cluster_id']}, Prediction={pred['prediction']}")

Testing with STRING labels...

CLUSTERING PIPELINE (Sliding Window) - Rule: Cluster Only (BATCH OPTIMIZED)
Stats: ['mean', 'std', 'min', 'max'], Window: 100, Features: SynAck, AckDat, TcpRtt

🔵 Encoding labels: ['HTTP' 'HTTPS' 'DNS']
Label mapping: {'DNS': np.int64(0), 'HTTP': np.int64(1), 'HTTPS': np.int64(2)}
Classes: 3

Data: 2000 samples, Classes: 3
  Class 0 (DNS): 582 (29.1%)
  Class 1 (HTTP): 1004 (50.2%)
  Class 2 (HTTPS): 414 (20.7%)

Train: 1599, Test: 401
Fixed features: ['SynAck', 'AckDat', 'TcpRtt'], Window: 100, Stats: ['mean', 'std', 'min', 'max']
Fixed features: ['SynAck', 'AckDat', 'TcpRtt'], Window: 100, Stats: ['mean', 'std', 'min', 'max']
Label coverage: target=1000, copied=1925

Classification features: 10 (excluding ['SynAck', 'AckDat', 'TcpRtt'])
Cluster 0: Trained on 262 samples, 3 classes
Cluster 1: Trained on 482 samples, 3 classes
Cluster 2: Trained on 855 samples, 3 classes

Evaluating test data (Rule: Cluster Only (BATCH OPTIMIZED))...

TEST RESULTS (Rule: 

In [7]:
train_data = 'DATASETS/CDR-MLC/scale_0.001/Short/CDR_MLC_Shuffle.csv'
test_data = 'DATASETS/CDR-MLC/scale_0.001/Long/CDR_MLC_Shuffle.csv'


results = run_pipeline_from_two_files(
    train_file=train_data,
    test_file=test_data,
    n_clusters=3,
    window_size=3,
    clustering_stats=['mean', 'median', 'std', 'min', 'max'],
    use_at_least_one_rule = False
)


Train file: DATASETS/CDR-MLC/scale_0.001/Short/CDR_MLC_Shuffle.csv, Shape: (329726, 36)
Test file: DATASETS/CDR-MLC/scale_0.001/Long/CDR_MLC_Shuffle.csv, Shape: (1417001, 36)
CLUSTERING PIPELINE (Sliding Window) - Rule: Cluster Only (BATCH OPTIMIZED)
Stats: ['mean', 'median', 'std', 'min', 'max'], Window: 3, Features: SynAck, AckDat, TcpRtt

🔵 Encoding labels: ['HTTP' 'VIDEO' 'SFTP' 'SMTP' 'SSH']
Label mapping: {'HTTP': np.int64(0), 'SFTP': np.int64(1), 'SMTP': np.int64(2), 'SSH': np.int64(3), 'VIDEO': np.int64(4)}
Classes: 5

Data: 329726 samples, Classes: 5
  Class 0 (HTTP): 178911 (54.3%)
  Class 1 (SFTP): 69374 (21.0%)
  Class 2 (SMTP): 27775 (8.4%)
  Class 3 (SSH): 16593 (5.0%)
  Class 4 (VIDEO): 37073 (11.2%)

Train: 329726, Test: 1417001
Fixed features: ['SynAck', 'AckDat', 'TcpRtt'], Window: 3, Stats: ['mean', 'median', 'std', 'min', 'max']
Fixed features: ['SynAck', 'AckDat', 'TcpRtt'], Window: 3, Stats: ['mean', 'median', 'std', 'min', 'max']
Label coverage: target=1000, copi

In [8]:
train_data = 'DATASETS/CDR-MLC/scale_0.001/Long/CDR_MLC_Shuffle.csv'
test_data = 'DATASETS/CDR-MLC/scale_0.001/Short/CDR_MLC_Shuffle.csv'


results = run_pipeline_from_two_files(
    train_file=train_data,
    test_file=test_data,
    n_clusters=3,
    window_size=3,
    clustering_stats=['mean', 'median', 'std', 'min', 'max'],
    use_at_least_one_rule = False
)


Train file: DATASETS/CDR-MLC/scale_0.001/Long/CDR_MLC_Shuffle.csv, Shape: (1417001, 36)
Test file: DATASETS/CDR-MLC/scale_0.001/Short/CDR_MLC_Shuffle.csv, Shape: (329726, 36)
CLUSTERING PIPELINE (Sliding Window) - Rule: Cluster Only (BATCH OPTIMIZED)
Stats: ['mean', 'median', 'std', 'min', 'max'], Window: 3, Features: SynAck, AckDat, TcpRtt

🔵 Encoding labels: ['SFTP' 'SSH' 'VIDEO' 'HTTP' 'SMTP']
Label mapping: {'HTTP': np.int64(0), 'SFTP': np.int64(1), 'SMTP': np.int64(2), 'SSH': np.int64(3), 'VIDEO': np.int64(4)}
Classes: 5

Data: 1417001 samples, Classes: 5
  Class 0 (HTTP): 420919 (29.7%)
  Class 1 (SFTP): 249568 (17.6%)
  Class 2 (SMTP): 334204 (23.6%)
  Class 3 (SSH): 82988 (5.9%)
  Class 4 (VIDEO): 329322 (23.2%)

Train: 1417001, Test: 329726
Fixed features: ['SynAck', 'AckDat', 'TcpRtt'], Window: 3, Stats: ['mean', 'median', 'std', 'min', 'max']
Fixed features: ['SynAck', 'AckDat', 'TcpRtt'], Window: 3, Stats: ['mean', 'median', 'std', 'min', 'max']
Label coverage: target=1000,

In [28]:
train_data = 'DATASETS/CDR-MLC/Long_CDR_MLC_Shuffle.csv'
test_data = 'DATASETS/CDR-MLC/CDR_MLC_Shuffle.csv'


results = run_pipeline_from_two_files(
    train_file=train_data,
    test_file=test_data,
    n_clusters=3,
    window_size=3,
    clustering_stats=['mean', 'median', 'std', 'min', 'max'],
    use_at_least_one_rule=True
)


Train file: DATASETS/CDR-MLC/Long_CDR_MLC_Shuffle.csv, Shape: (1417001, 36)
Test file: DATASETS/CDR-MLC/CDR_MLC_Shuffle.csv, Shape: (329726, 36)
CLUSTERING PIPELINE (Sliding Window) - Rule: At Least One
Stats: ['mean', 'median', 'std', 'min', 'max'], Window: 3, Features: SynAck, AckDat, TcpRtt

🔵 Encoding labels: <StringArray>
['SFTP', 'SSH', 'VIDEO', 'HTTP', 'SMTP']
Length: 5, dtype: str
Label mapping: {'HTTP': np.int64(0), 'SFTP': np.int64(1), 'SMTP': np.int64(2), 'SSH': np.int64(3), 'VIDEO': np.int64(4)}
Classes: 5

Data: 1417001 samples, Classes: 5
  Class 0 (HTTP): 420919 (29.7%)
  Class 1 (SFTP): 249568 (17.6%)
  Class 2 (SMTP): 334204 (23.6%)
  Class 3 (SSH): 82988 (5.9%)
  Class 4 (VIDEO): 329322 (23.2%)

Train: 1417001, Test: 329726
Fixed features: ['SynAck', 'AckDat', 'TcpRtt'], Window: 3, Stats: ['mean', 'median', 'std', 'min', 'max']
Fixed features: ['SynAck', 'AckDat', 'TcpRtt'], Window: 3, Stats: ['mean', 'median', 'std', 'min', 'max']
Label coverage: target=1000, copied=

In [ ]:
train_data = 'DATASETS/CDR-MLC/level_1 .csv'
test_data = 'DATASETS/CDR-MLC/level_3.csv'


results = run_pipeline_from_two_files(
    train_file=train_data,
    test_file=test_data,
    n_clusters=3,
    window_size=3,
    clustering_stats=['mean', 'median', 'std', 'min', 'max'],
    use_at_least_one_rule=False
)

Train file: DATASETS/CDR-MLC/level_1.csv, Shape: (68079, 36)
Test file: DATASETS/CDR-MLC/level_3.csv, Shape: (60482, 36)
CLUSTERING PIPELINE (Sliding Window) - Rule: Cluster Only (BATCH OPTIMIZED)
Stats: ['mean', 'median', 'std', 'min', 'max'], Window: 3, Features: SynAck, AckDat, TcpRtt

🔵 Encoding labels: <StringArray>
['HTTP', 'SFTP', 'SMTP', 'SSH', 'VIDEO']
Length: 5, dtype: str
Label mapping: {'HTTP': np.int64(0), 'SFTP': np.int64(1), 'SMTP': np.int64(2), 'SSH': np.int64(3), 'VIDEO': np.int64(4)}
Classes: 5

Data: 68079 samples, Classes: 5
  Class 0 (HTTP): 39798 (58.5%)
  Class 1 (SFTP): 11422 (16.8%)
  Class 2 (SMTP): 6267 (9.2%)
  Class 3 (SSH): 3127 (4.6%)
  Class 4 (VIDEO): 7465 (11.0%)

Train: 68079, Test: 60482
Fixed features: ['SynAck', 'AckDat', 'TcpRtt'], Window: 3, Stats: ['mean', 'median', 'std', 'min', 'max']
Fixed features: ['SynAck', 'AckDat', 'TcpRtt'], Window: 3, Stats: ['mean', 'median', 'std', 'min', 'max']
Label coverage: target=1000, copied=4281

Classificatio

In [ ]:

train_data = 'DATASETS/ISCX-VPN/ISCX-VPN.csv'


results = run_pipeline_from_file(
    train_data,
    n_clusters=3,
    window_size=3,
    clustering_stats=['mean', 'median', 'std', 'min', 'max'],
    use_at_least_one_rule=False
)